# Course audio re-render — Don's voice (EN + ES)

Run each cell **in order, top to bottom**. Shift+Enter on a cell runs it and advances to the next.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.

Total time: ~3-5 hours on a T4. Don't close this browser tab while a cell is running.


## 1. Confirm GPU

Output should mention `Tesla T4` (or similar). If it errors, fix the Runtime type first.

In [ ]:
!nvidia-smi

## 2. Install everything (~2 min)

Node 20 (the render script needs modern Node), F5-TTS for voice cloning, jieba (an undeclared F5 dep on fresh Colab images), and huggingface_hub for fetching the Spanish model.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs -qq > /dev/null
!node --version
!pip install -q f5-tts jieba huggingface_hub

## 3. Clone the repo + patch a known F5 quirk

`torch.xpu` (Intel GPU) is hardcoded in F5 but isn't shipped with Colab's CUDA build. The `sed` line below makes the check safe.

In [ ]:
!rm -rf /content/potentially-profitable
!git clone -b claude/upbeat-sagan-3PnfC --depth 1 https://github.com/Donwonmagic/potentially-profitable.git /content/potentially-profitable
%cd /content/potentially-profitable

!find /usr/local/lib/python*/dist-packages/f5_tts -name '*.py' -exec sed -i 's/torch\.xpu\.is_available()/(hasattr(torch, "xpu") and torch.xpu.is_available())/g' {} +
print('repo cloned + F5 patched')

## 4. Download the Spanish model + verify Don's Spanish reference

`ES_CKPT` and `ES_VOCAB` are the F5-Spanish fine-tune (~1.3 GB; cached on second run). The Spanish reference clip should already be in the cloned repo.

In [ ]:
import os
from huggingface_hub import hf_hub_download

ES_CKPT  = hf_hub_download('jpgallegoar/F5-Spanish', 'model_1200000.safetensors')
ES_VOCAB = hf_hub_download('jpgallegoar/F5-Spanish', 'vocab.txt')
ES_REF   = '/content/potentially-profitable/scripts/voice-refs/don-reference.es.m4a'

print('ckpt  ->', ES_CKPT)
print('vocab ->', ES_VOCAB)
print('don ref present:', os.path.isfile(ES_REF))

## 5. (Skip unless Cell 4 said "don ref present: False")

If the Spanish reference is missing, this cell pops a file picker — upload `don-reference.es.m4a` from your Mac and it'll land in the right path.

In [ ]:
import os
TARGET = '/content/potentially-profitable/scripts/voice-refs/don-reference.es.m4a'

if os.path.isfile(TARGET):
    print('already present — skip this cell')
else:
    from google.colab import files
    os.makedirs(os.path.dirname(TARGET), exist_ok=True)
    up = files.upload()  # pick don-reference.es.m4a
    for name, data in up.items():
        with open(TARGET, 'wb') as f:
            f.write(data)
    print(f'saved {os.path.getsize(TARGET)} bytes -> {TARGET}')

## 6. Wipe the old (rushed) audio so the re-render actually runs

The renderer skips lessons whose `audio.mp3` already exists. Since we cloned a branch that has the previous render committed, we delete those files so every lesson re-renders from scratch with the new pacing + breath + pronunciation fixes.

In [ ]:
import subprocess
out = subprocess.check_output(
    ['find', 'course', 'es/course', '-maxdepth', '5',
     '(', '-name', 'audio*.mp3', '-o', '-name', 'audio*.json', ')',
     '-delete', '-print'],
    cwd='/content/potentially-profitable'
).decode()
print(f'deleted {len(out.splitlines())} files')

## 7. Render — the long one (~3-5 hours)

This is the cell that does the work. F5 base model loads first (~20s, ~1.3 GB download), then 20 English lessons render. After English finishes, F5-Spanish loads (another ~1.3 GB) and 20 Spanish lessons render.

Each chunk should take 2-4 s on the T4. Progress lines look like `0 3.179 2.760` (chunk id, audio seconds, wall-clock seconds).

**Do not close this tab while this cell runs.** Colab will idle-disconnect after ~90 min of no activity in the tab; check on it occasionally.

In [ ]:
!node scripts/render-course-batch.mjs --run --native-only --engine f5 --f5-languages en,es --f5-ckpt-es {ES_CKPT} --f5-vocab-es {ES_VOCAB} --f5-model-name-es F5TTS_Base

## 8. Zip the audio and download

Bundles every fresh MP3 + JSON manifest into one zip and triggers a browser download. Drop it into `~/Downloads` on your Mac when prompted.

In [ ]:
import glob, os, zipfile

out = '/content/course-audio.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for pattern in ['course/**/audio*.mp3', 'course/**/audio*.json',
                    'es/course/**/audio*.mp3', 'es/course/**/audio*.json']:
        for f in sorted(glob.glob(pattern, recursive=True)):
            z.write(f)
            print('+', f)

size_mb = os.path.getsize(out) // 1024 // 1024
print(f'\nzipped {size_mb} MB -> {out}')

from google.colab import files
files.download(out)

## 9. Back on your Mac — commit and push

In your Mac terminal:

```bash
cd ~/potentially-profitable
git checkout claude/upbeat-sagan-3PnfC
git pull
unzip -o ~/Downloads/course-audio.zip
git add course es/course
git commit -m "Re-render course audio with slower pace + natural breaths + pronunciation dict"
git push
```

That's it. The runtime player picks up the new MP3s automatically — no HTML edits needed.